# weight-decay-l2-add composite — cx13: L2 weight decay folded into the SGD momentum buffer

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `weight-decay-l2-add`, `momentum-buffer-update`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "weight-decay-l2-add"
DD_ATOM_IDS = ["weight-decay-l2-add", "momentum-buffer-update"]
DD_SUBTOPICS = ["Optimizer: Weight decay L2", "Optimizer: Momentum buffer"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

PyTorch's `torch.optim.SGD(momentum=mu, weight_decay=lam)` does NOT apply weight decay as a separate post-step. Instead it ADDS `lam * theta` to the gradient BEFORE the momentum buffer is updated, so the buffer carries WD-augmented gradients through every step.

**Atom A — weight-decay-l2-add.** The L2 trick: `grad <- grad + lam * theta`. This is equivalent (only for vanilla SGD without momentum) to adding `(lam/2) * ||theta||^2` to the loss. With momentum, it is NOT mathematically identical to L2 regularization any more — but it is what PyTorch's SGD does, and the test cross-checks against it.

**Atom B — momentum-buffer-update.** `b <- mu * b + g`. The momentum buffer accumulates the (now WD-augmented) gradient with an EMA-like decay.

**Anatomy of one step.**
```python
for p in params:
    g = p.grad
    if lam != 0: g = g + lam * p          # Atom A: fold WD into the grad.
    if mu != 0:
        if buf[p] is None: buf[p] = g.clone()
        else: buf[p].mul_(mu).add_(g)     # Atom B: b = mu*b + g.
        g = buf[p]
    p.data.add_(g, alpha=-lr)             # update theta.
```

**Why both atoms together.** Cross-checks against `torch.optim.SGD` only line up if the WD is added to `g` BEFORE the buffer update. Doing it after (or as a separate `theta *= (1 - lr*lam)` step like AdamW) gives a different trajectory.

### Composite Exercise — L2 weight decay folded into the SGD momentum buffer

**Atoms exercised together**: `weight-decay-l2-add`, `momentum-buffer-update`

Implement `cx13_sgd_step(params, lr, momentum, weight_decay, buffers)`:

- `params`: list of `t.Tensor` with `.grad` populated and `.data` to be updated in-place.
- `buffers`: dict mapping `id(param) -> t.Tensor or None`. Updated in-place. (Caller supplies an initial `{id(p): None for p in params}`.)
- Steps per param:
  1. Read `g = p.grad`.
  2. If `weight_decay != 0`: `g = g + weight_decay * p.data` (the L2 fold; do NOT mutate `p.grad`).
  3. If `momentum != 0`: if `buffers[id(p)] is None` initialise it to `g.clone()`, else do `buffers[id(p)].mul_(momentum).add_(g)`; then set `g = buffers[id(p)]`.
  4. `p.data.add_(g, alpha=-lr)`.

The test cross-checks the trajectory against `torch.optim.SGD(lr, momentum, weight_decay)`.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx13_sgd_step(params, lr, momentum, weight_decay, buffers):
    """In-place one-step SGD with momentum + L2 weight decay."""
    raise NotImplementedError

def _test_cx13():
    # Case A: matches torch.optim.SGD over a 4-step trajectory.
    t.manual_seed(0)
    lr, mu, lam = 0.05, 0.9, 0.01
    x_ref = t.randn(6, requires_grad=True)
    x_mine = x_ref.detach().clone().requires_grad_(True)
    ref_opt = t.optim.SGD([x_ref], lr=lr, momentum=mu, weight_decay=lam)
    buffers = {id(x_mine): None}

    for step in range(4):
        loss_ref = (x_ref * t.arange(1.0, 7.0)).pow(2).sum()
        ref_opt.zero_grad()
        loss_ref.backward()
        ref_opt.step()

        loss_mine = (x_mine * t.arange(1.0, 7.0)).pow(2).sum()
        if x_mine.grad is not None:
            x_mine.grad.zero_()
        loss_mine.backward()
        cx13_sgd_step([x_mine], lr=lr, momentum=mu, weight_decay=lam, buffers=buffers)

        assert t.allclose(x_mine.data, x_ref.data, atol=1e-5), (
            f'step {step}: mine={x_mine.data}, ref={x_ref.data}'
        )

    # Case B: with mu=0, just gradient + WD.
    t.manual_seed(1)
    p = t.tensor([1.0, 2.0, 3.0], requires_grad=True)
    p.grad = t.tensor([0.1, 0.1, 0.1])
    before = p.data.clone()
    buf = {id(p): None}
    cx13_sgd_step([p], lr=0.1, momentum=0.0, weight_decay=0.5, buffers=buf)
    expected = before - 0.1 * (t.tensor([0.1, 0.1, 0.1]) + 0.5 * before)
    assert t.allclose(p.data, expected, atol=1e-6), f'mu=0 path: got {p.data}, want {expected}'

    # Case C: buffer is actually updated in-place (not replaced).
    t.manual_seed(2)
    p = t.tensor([1.0, 1.0], requires_grad=True)
    p.grad = t.tensor([0.2, 0.2])
    buf = {id(p): None}
    cx13_sgd_step([p], lr=0.01, momentum=0.9, weight_decay=0.0, buffers=buf)
    assert buf[id(p)] is not None, 'buffer not initialised on first step'
    first_buf = buf[id(p)]
    p.grad = t.tensor([0.3, 0.3])
    cx13_sgd_step([p], lr=0.01, momentum=0.9, weight_decay=0.0, buffers=buf)
    assert buf[id(p)] is first_buf, 'buffer must be updated in-place across steps, not replaced'
    expected_buf = 0.9 * t.tensor([0.2, 0.2]) + t.tensor([0.3, 0.3])
    assert t.allclose(buf[id(p)], expected_buf, atol=1e-6), (
        f'buffer update wrong: got {buf[id(p)]}, want {expected_buf}'
    )

    # Case D: WD must NOT mutate p.grad (only the local `g`).
    p = t.tensor([5.0, 5.0], requires_grad=True)
    g_in = t.tensor([0.0, 0.0])
    p.grad = g_in
    buf = {id(p): None}
    cx13_sgd_step([p], lr=0.01, momentum=0.0, weight_decay=0.1, buffers=buf)
    assert t.allclose(p.grad, t.tensor([0.0, 0.0])), (
        f'WD must not write back into p.grad; got {p.grad}'
    )
    _dd_passed.add('cx13')

_test_cx13()

<details><summary>Show solution — cx13</summary>

```python
def cx13_sgd_step(params, lr, momentum, weight_decay, buffers):
    for p in params:
        # Atom A (weight-decay-l2-add): fold lam*theta into the grad BEFORE the buffer update.
        g = p.grad
        if weight_decay != 0:
            g = g + weight_decay * p.data   # new tensor — leaves p.grad alone.
        # Atom B (momentum-buffer-update): b <- mu*b + g, in-place.
        if momentum != 0:
            buf = buffers[id(p)]
            if buf is None:
                buffers[id(p)] = g.clone().detach()
            else:
                buf.mul_(momentum).add_(g)
            g = buffers[id(p)]
        # In-place parameter update.
        p.data.add_(g, alpha=-lr)
```

Two ordering traps: (1) folding WD AFTER the buffer update would shift the WD contribution off by one step and diverge from `torch.optim.SGD`. (2) Initialising the buffer with the WD-augmented `g.clone()` is what PyTorch does, so the first step uses `g` directly (not `mu*0 + g`). Cloning is essential — assigning `buffers[id(p)] = g` would alias the buffer to whatever tensor `g` is currently bound to, and the next step's reassignment would silently drop the running state.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx13'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx13',
        'subtopics': ["Optimizer: Weight decay L2", "Optimizer: Momentum buffer"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()